In [ ]:
from pathlib import Path
import sys
import os

current_directory = Path(os.getcwd())
print(current_directory)
root_directory = current_directory.parent.parent
print(root_directory)

sys.path.append(str(root_directory))

In [ ]:
from pathlib import Path
import typing as t
import re
from pprint import pprint
import coq_serapy as c
import json
import pickle
import random

from src.config import CONFIG
from src.coq_serapy_util import LemmaLocation, read_commands, get_section_name_from_command
from src.dataset import Example

In [ ]:
WIGDERSON_DIR = Path(CONFIG.ROOT_DIR) / "coq-projects/coq-wigderson"
WIGDERSON_DIR

In [ ]:
# cd wigderson_dir && find . -name '*.v' -exec grep -o 'Qed.' {} + | wc -l
command = f"cd {WIGDERSON_DIR} && find . -name '*.v' -exec grep -o 'Qed.' {{}} + | wc -l"
num_qeds = int(os.popen(command).read())
num_qeds

In [ ]:
command = f"cd {WIGDERSON_DIR} && find . -name '*.v' -exec grep -o 'Admitted.' {{}} + | wc -l"
num_admitted = int(os.popen(command).read())
num_admitted

In [ ]:
command = f"cd {WIGDERSON_DIR} && find . -name '*.v' -exec grep -o 'Defined.' {{}} + | wc -l"
num_defined = int(os.popen(command).read())
num_defined

In [ ]:
command = f"cd {WIGDERSON_DIR} && find . -name '*.v' -exec grep -o 'Proof.' {{}} + | wc -l"
num_proofs = int(os.popen(command).read())
num_proofs

In [ ]:
COQ_PROJECT_PATH = WIGDERSON_DIR / "_CoqProject"
# filenames = all files after the first
filenames = [line.strip() for line in COQ_PROJECT_PATH.read_text().split("\n")[1:] if line]
filenames

In [ ]:
file_paths = [WIGDERSON_DIR / filename for filename in filenames]
file_paths

In [ ]:
file_to_num_qed: t.Dict[Path, int] = {
    file_path: int(os.popen(f"grep -o 'Qed.' {file_path} | wc -l").read())
    for file_path in file_paths
}
file_to_num_qed

In [ ]:
file_to_num_admitted: t.Dict[Path, int] = {
    file_path: int(os.popen(f"grep -o 'Admitted.' {file_path} | wc -l").read())
    for file_path in file_paths
}
file_to_num_admitted

In [ ]:
file_to_num_defined: t.Dict[Path, int] = {
    file_path: int(os.popen(f"grep -o 'Defined.' {file_path} | wc -l").read())
    for file_path in file_paths
}
file_to_num_defined

# get the nth proof in a file

In [ ]:
TEST_FILE = file_paths[0]
TEST_NUM_QED = file_to_num_qed[TEST_FILE]
TEST_FILE, TEST_NUM_QED

In [ ]:
TEST_IDX = 5

In [ ]:
TEST_FILE_COMMANDS = read_commands(TEST_FILE.read_text())
TEST_FILE_COMMANDS[0:5]

In [ ]:
num_lemmas = 0
for command in TEST_FILE_COMMANDS:
    try:
        lemma_name = c.coq_util.lemma_name_from_statement(command[0])
        if lemma_name.strip() == "":
            continue
        if "Theorem" not in command[0] and "Lemma" not in command[0]:
            continue
        num_lemmas += 1
        print(lemma_name, num_lemmas)
    except:
        continue

In [ ]:
# TODO: factor this into a utility in coq serapy utisl

section_regex = re.compile(r"Section\s+(\w+)")
end_section_regex = re.compile(r"End\s+(\w+)")
lemma_regex = re.compile(r"(Example|Lemma|Theorem)\s+(\w+)\s*(\{\w+\})?\s*:([\s\S]*)\.")

section_names: t.List[str] = []
lemmas: t.List[LemmaLocation] = []
for command, line in TEST_FILE_COMMANDS:
    print("processing", command)
    
    section_match = section_regex.match(command)
    if section_match is not None:
        print("section match")
        section_name = section_match.group(1)
        section_names.append(section_name)
        continue

    end_section_match = end_section_regex.match(command)
    if end_section_match is not None:
        print("end section match")
        section_name = end_section_match.group(1)
        if section_name != section_names[-1]:
            raise ValueError(f"Section name mismatch: {section_name} != {section_names[-1]}")
        section_names.pop()
        continue

    try:
        lemma_name = c.coq_util.lemma_name_from_statement(command)
        if lemma_name.strip() == "":
            continue
        if "Theorem" not in command and "Lemma" not in command:
            continue
        print("lemma match", lemma_name)
        lemma_location = LemmaLocation(
            project_name="coq-wigderson",
            file_name=TEST_FILE.name,
            lemma_name=lemma_name,
            section_names=section_names.copy(),
            coq_version="8.13"
        )
        lemmas.append(lemma_location)
    except:
        pass

len(lemmas)


In [ ]:
lemmas

In [ ]:
section_regex = re.compile(r"Section\s+(\w+)")
end_section_regex = re.compile(r"End\s+(\w+)")
lemma_regex = re.compile(r"(Example|Lemma|Theorem|Function)\s+(\w+)[\s\S]*(?!:=):([\s\S]*)\.")
# lemma_regex = re.compile(r"(Example|Lemma|Theorem)\s+(\w+)\s*(\{\w+\})?\s*(\(.*\))?\s*:([\s\S]*)\.")

def get_all_lemma_locations(file: Path) -> t.List[Example]:
    commands = read_commands(file.read_text())
    
    print(file)

    section_names: t.List[str] = []
    lemmas: t.List[Example] = []
    in_proof = False
    for command, line in commands:
        command = c.coq_util.kill_comments(command).strip()
        section_match = section_regex.match(command)
        if section_match is not None:
            section_name = section_match.group(1)
            section_names.append(section_name)
            continue

        end_section_match = end_section_regex.match(command)
        if end_section_match is not None:
            section_name = end_section_match.group(1)
            if section_name != section_names[-1]:
                raise ValueError(f"Section name mismatch: {section_name} != {section_names[-1]}")
            section_names.pop()
            continue

        lemma_match = lemma_regex.match(command)
        if lemma_match is not None:
            print(command)
            lemma_name = lemma_match.group(2)
            lemma_location = LemmaLocation(
                project_name="coq-wigderson",
                file_name=file.name,
                lemma_name=lemma_name,
                section_names=section_names.copy(),
                coq_version="8.13"
            )
            example = Example(
                gold_standard_proof="",
                proposition_command=command,
                location=lemma_location
            )
            lemmas.append(example)

        # try:
        #     lemma_name = c.coq_util.lemma_name_from_statement(command)
        #     if lemma_name.strip() == "":
        #         continue
        #     if "Theorem" not in command and "Lemma" not in command and "Example" not in command:
        #         continue
        #     lemma_location = LemmaLocation(
        #         project_name="coq-wigderson",
        #         file_name=file.name,
        #         lemma_name=lemma_name,
        #         section_names=section_names.copy(),
        #         coq_version="8.13"
        #     )
        #     lemmas.append(lemma_location)
        # except:
        #     pass

        if command == 'Proof.':
            in_proof = True

        if in_proof and (command == 'Admitted.' or command == 'Abort.'):
            print("admitted or abort.")
            # remove the last lemma location
            lemmas.pop()
        
        if command == 'Qed.' or command == 'Defined.' or command == 'Admitted.' or command == 'Abort.':
            in_proof = False
    return lemmas

In [ ]:
LEMMAS: t.Dict[Path, t.List[Example]] = {
    file: get_all_lemma_locations(file)
    for file in file_paths
}
LEMMAS

In [ ]:
NUM_LEMMAS = {
    file: len(lemmas)
    for file, lemmas in LEMMAS.items()
}
NUM_LEMMAS

# Sample lemmas

In [ ]:
from src.dataset.coq_wigderson import COQ_WIGDERSON_DEV_SAMPLED_DATASET

LEMMAS_NOT_IN_DEV = {
    file: [lemma for lemma in lemmas if lemma not in COQ_WIGDERSON_DEV_SAMPLED_DATASET]
    for file, lemmas in LEMMAS.items()
}

NUM_LEMMAS_NOT_IN_DEV = {
    file: len(lemmas)
    for file, lemmas in LEMMAS_NOT_IN_DEV.items()
}
NUM_LEMMAS_NOT_IN_DEV

In [ ]:
TOTAL_LEMMAS_NOT_IN_DEV = sum(NUM_LEMMAS_NOT_IN_DEV.values())
TOTAL_LEMMAS_NOT_IN_DEV

In [ ]:
TOTAL_LEMMAS = sum(NUM_LEMMAS.values())
TOTAL_LEMMAS

In [ ]:
CONCATENATED_LEMMAS = [
    lemma
    for file, lemmas in LEMMAS_NOT_IN_DEV.items()
    for lemma in lemmas
]
len(CONCATENATED_LEMMAS)

In [ ]:
def sample(n: int) -> t.List[Example]:
    return random.sample(CONCATENATED_LEMMAS, n)

TEST = sample(100)

In [ ]:
TEST

# serialize

In [ ]:
import pickle

W_TEST_SAMPLED_FILE = Path(CONFIG.ROOT_DIR) / "data/COQ_WIGDERSON_TEST_SAMPLED_DATASET.pkl"
with W_TEST_SAMPLED_FILE.open("wb") as f:
    pickle.dump(TEST, f)

In [ ]:
# dump sorted example names as a csv
import csv

EXAMPLE_NAMES = sorted([example.name for example in TEST])

W_TEST_SAMPLED_CSV_FILE = Path(CONFIG.ROOT_DIR) / "data/COQ_WIGDERSON_TEST_SAMPLED_DATASET.csv"
with W_TEST_SAMPLED_CSV_FILE.open("w") as f:
    writer = csv.writer(f)
    writer.writerow(["example_name"])
    for example_name in EXAMPLE_NAMES:
        writer.writerow([
            example_name
        ])